# 18 — Production Quality, CI, and Deployment

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
import json
import sqlite3

# Provided for this lesson.
from collections.abc import Callable

from lesson import Exercises, ___, run

check = Exercises('advanced/18', {1: '84a9d7da12fda46c', 2: 'b35a4464d573e9bf', 3: '4c25f3428c03536b', 4: 'b8a67c271fd2b282', 5: 'ab728609c14d40d2', 6: 'b1794fdb486f32f0', 7: '9819ce8a114c851c'}, practice='advanced_04_production_systems', independent='migrate')

## 1. Idempotent operations

- Retries and duplicate clicks happen. Store the result under an idempotency key and return it for repeats.
- Store only after the action succeeds, so a failed attempt can be retried.
- Results must be JSON-serialisable to keep them in the database.

In [ ]:
import sqlite3, json
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE idempotency (key TEXT PRIMARY KEY, result TEXT NOT NULL)")
runs = []
def execute(key, action):
    row = conn.execute("SELECT result FROM idempotency WHERE key = ?", (key,)).fetchone()
    if row:
        return json.loads(row[0])
    result = action()
    with conn:
        conn.execute("INSERT INTO idempotency VALUES (?, ?)", (key, json.dumps(result)))
    return result

print(execute("k", lambda: runs.append(1) or {"ok": True}), execute("k", lambda: runs.append(1)), len(runs))

**Exercise 1.** Same key twice: how many times does the action run?

In [ ]:
predicted = ___
check(1, predicted)

**Exercise 2.** Store the result as JSON text and read it back.

In [ ]:
import json
stored = json.dumps({"receipt": 7})
check(2, json.loads("" + ___))   # load the stored text

**Exercise 3.** If the action raises, should anything be stored? Write `True` or `False`.

In [ ]:
answer = ___
check(3, answer)

## 2. Schema migrations

- Number migrations and apply them in order, each in its own transaction.
- SQLite's `PRAGMA user_version` stores which migration ran last; start from there.
- Re-running must be a no-op: already-applied migrations are skipped.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
migrations = ["CREATE TABLE users (id INTEGER PRIMARY KEY)", "ALTER TABLE users ADD COLUMN email TEXT"]
for _ in range(2):   # run twice: the second pass does nothing
    version = conn.execute("PRAGMA user_version").fetchone()[0]
    for index in range(version, len(migrations)):
        with conn:
            conn.execute(migrations[index])
            conn.execute(f"PRAGMA user_version = {index + 1}")
print(conn.execute("PRAGMA user_version").fetchone()[0])

**Exercise 4.** At version 1 with 3 migrations, which indexes still need to run?

In [ ]:
version, migrations = 1, ["m0", "m1", "m2"]
pending = list(range(___, len(migrations)))
check(4, pending)

**Exercise 5.** What does `PRAGMA user_version` return on a fresh database?

In [ ]:
import sqlite3
version = sqlite3.connect(":memory:").execute("PRAGMA user_version").fetchone()[0]
predicted = ___
check(5, predicted)

## 3. CI and deployment

- CI runs the same checks on every change: tests, linting, type checks. A red build blocks the merge.
- Configuration comes from the environment (`os.environ.get("PORT", "8000")`), never hard-coded secrets.
- Deploy gradually, watch the metrics and keep a rollback ready.

In [ ]:
import os
port = int(os.environ.get("APP_PORT", "8000"))
debug = os.environ.get("APP_DEBUG", "false").lower() == "true"
print(port, debug)

**Exercise 6.** Read a setting with a default when it's unset.

In [ ]:
import os
os.environ.pop("APP_WORKERS", None)
workers = int(os.environ.get("APP_WORKERS", ___))
check(6, workers)   # default to "4"

**Exercise 7.** Where should an API key live: `"code"` or `"environment"`?

In [ ]:
answer = ___
check(7, answer)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `IdempotencyStore`

Run an action at most once per key, stored in SQLite.

In [ ]:
class IdempotencyStore:
    """SQLite-backed idempotency: execute(key, action) runs action() at most once per key
    and stores its JSON-serializable result; later calls return the stored result.

    If action() raises, nothing is stored, so a retry can run it again.
    """

    def __init__(self, conn: sqlite3.Connection):
        self.conn = conn
        with ___:
            conn.execute("CREATE TABLE IF NOT EXISTS idempotency (key TEXT PRIMARY KEY, result TEXT NOT NULL)")

    def execute(self, key: str, action: Callable[[], object]):
        row = self.conn.execute("SELECT result FROM idempotency WHERE key = ?", (key,)).fetchone()
        if ___:
            return ___
        result = action()
        with self.conn:
            self.conn.execute("INSERT INTO idempotency (key, result) VALUES (?, ?)", (key, json.dumps(result)))
        return result

In [ ]:
check('IdempotencyStore')

### Independent build — `migrate`

Starting at SQLite PRAGMA user_version (the number already applied), execute the remaining single-statement migrations in order. Commit each migration and its version increment atomically. Return the final version. Rerunning at that version changes nothing.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def migrate(conn: sqlite3.Connection, migrations: list[str]) -> int:
    raise NotImplementedError

In [ ]:
check('migrate')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Secrets should come from…

   - **a)** Environment or a secret store at runtime
   - **b)** The source code
   - **c)** Git history

**2.** A readiness check answers…

   - **a)** Can it serve traffic right now (dependencies OK)?
   - **b)** Has it started?
   - **c)** Is the process alive?

**3.** A useful CI pipeline runs…

   - **a)** Only manual tests
   - **b)** Only on release day
   - **c)** Syntax checks and tests on every change

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/18', {1: '17e8994c8384', 2: 'b1ff8b23a6aa', 3: '00a05fd10354'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [19 — Advanced Python Review and Capstone Preparation](19_advanced_review_capstone.ipynb)